In [3]:
import numpy as np
from PIL import Image

img = Image.open("im1.jpg").convert("RGB")
arr = np.array(img, dtype=np.float64)         
H, W = arr.shape[:2]

def fft2_channel(channel):
    F = np.fft.fft(channel, axis=1)   
    F = np.fft.fft(F,       axis=0)  
    return F

spectra = [fft2_channel(arr[:, :, c]) for c in range(3)]   

def spectrum_to_uint8(F):
    shifted   = np.fft.fftshift(F)
    magnitude = np.log1p(np.abs(shifted))
    lo, hi    = magnitude.min(), magnitude.max()
    norm      = (magnitude - lo) / (hi - lo) * 255.0 if hi > lo else magnitude
    return np.round(norm).astype(np.uint8)  

channels_vis = [spectrum_to_uint8(F) for F in spectra]
out_img = Image.fromarray(np.stack(channels_vis, axis=2), mode="RGB")
out_img.save("out.jpg")

def remove_upper_half(F):
    F2 = F.copy()
    F2[H // 2:, :] = 0 
    F2[:, W // 2:] = 0 
    return F2

spectra_filtered = [remove_upper_half(F) for F in spectra]

def ifft2_channel(F):
    f = np.fft.ifft(F, axis=0)  
    f = np.fft.ifft(f, axis=1)  
    return np.real(f)

restored_channels = [ifft2_channel(F) for F in spectra_filtered]
restored_arr = np.stack(restored_channels, axis=2)
restored_arr = np.clip(restored_arr, 0, 255).astype(np.uint8)

out2_img = Image.fromarray(restored_arr, mode="RGB")
out2_img.save("out2.jpg")

In [ ]:
import wave
import struct
import scipy.io.wavfile as wavfile

def read_wav(path):
    """Читает WAV, возвращает (samples int16/int32, params)."""
    wav = wave.open(path, mode="r")
    params = wav.getparams()
    nchannels, sampwidth, framerate, nframes = params[:4]
    content = wav.readframes(nframes)
    wav.close()
    types = {1: np.int8, 2: np.int16, 4: np.int32}
    samples = np.frombuffer(content, dtype=types[sampwidth]).copy()
    return samples, params

def write_wav(path, samples, params):
    """Записывает samples (int16) в WAV с теми же параметрами."""
    nchannels, sampwidth, framerate, nframes, comptype, compname = params
    out = wave.open(path, mode="wb")
    out.setparams(params)
    newframes = struct.pack('<' + str(len(samples)) + 'h', *samples.astype(np.int16))
    out.writeframes(newframes)
    out.close()

samples, params = read_wav("in10.wav")
nchannels, sampwidth, framerate, nframes = params[:4]

data = samples[0::nchannels].astype(np.float64)
N = len(data)

spectrum = np.fft.fft(data)
freqs    = np.fft.fftfreq(N, d=1.0 / framerate)


spectrum_noisy = spectrum.copy()
noise_power = np.max(np.abs(spectrum)) * 0.3

rng = np.random.default_rng(42)
low_idx = np.where(np.abs(freqs) <= 300)[0]

noise_real = rng.uniform(-noise_power, noise_power, size=len(low_idx))
noise_imag = rng.uniform(-noise_power, noise_power, size=len(low_idx))
spectrum_noisy[low_idx] += noise_real + 1j * noise_imag

data_noisy = np.real(np.fft.ifft(spectrum_noisy))

peak = 256 ** sampwidth / 2
data_noisy_int = np.clip(data_noisy, -peak, peak - 1).astype(np.int16)

if nchannels == 2:
    interleaved = np.empty(len(data_noisy_int) * 2, dtype=np.int16)
    interleaved[0::2] = data_noisy_int
    interleaved[1::2] = data_noisy_int
    write_wav("out1.wav", interleaved, params)
else:
    write_wav("out1.wav", data_noisy_int, params)

spectrum_filtered = spectrum_noisy.copy()
low_mask = np.abs(freqs) < 300
spectrum_filtered[low_mask] = 0
n_removed = np.sum(low_mask)




data_filtered = np.real(np.fft.ifft(spectrum_filtered))
data_filtered_int = np.clip(data_filtered, -peak, peak - 1).astype(np.int16)

if nchannels == 2:
    interleaved2 = np.empty(len(data_filtered_int) * 2, dtype=np.int16)
    interleaved2[0::2] = data_filtered_int
    interleaved2[1::2] = data_filtered_int
    write_wav("out2.wav", interleaved2, params)
else:
    write_wav("out2.wav", data_filtered_int, params)